# Delivering Outputs to Object Storage

## What you'll learn

- Send a Modal-executed command op's outputs to an allowed S3-compatible
  bucket prefix with `output_store`
- Follow the delivery: the worker uploads a tarball under your prefix
  and the client fetches it through a presigned URL
- Verify and clean up the delivered objects

**Prerequisites:** [Running on Modal](04-modal-execution.ipynb), an S3-compatible
bucket, and credentials that can write and delete objects under a tutorial prefix.
This optional tutorial makes billable cloud calls.
**Estimated time:** 10 minutes.
**GPU required:** No


:::{note}
Complete the deployment setup below before running the notebook. It uses the
`R2WaitTool` class in the neighboring `r2_wait.py` file, a subclass of the bundled
`WaitTool` with its own endpoint and output policy. Run the notebook from this
source checkout so that module is importable.
:::

## Deliver outputs to a bucket

The default endpoint response returns outputs inline, subject to a size limit.
An `output_store` sends the output archive to an allowed bucket prefix instead.
The client downloads the archive through a presigned URL and commits the results
to the pipeline as usual.

The allowed destination and worker credentials belong to the deployment. Each
call can choose a subdirectory within that destination. See
[Object-store output delivery](../../how-to-guides/deploying-tool-endpoints.md#object-store-output-delivery)
for the transport contract.

## One-time setup

Follow [Deploy the R2 wait example](../../how-to-guides/deploying-tool-endpoints.md#deploy-the-r2-wait-example).
Configure the `r2-artisan` Modal secret and deploy the `R2WaitTool` class
used below as `r2_wait_tool`, following that guide. This setup is separate from the live
integration tests.

Set these values in the source checkout’s `.env` file or your environment before
deployment and before starting the notebook:

```dotenv
AWS_ACCESS_KEY_ID=...
AWS_SECRET_ACCESS_KEY=...
ARTISAN_S3_ENDPOINT_URL=https://<account>.r2.cloudflarestorage.com
ARTISAN_S3_BUCKET=my-bucket
```

The notebook uses the local credentials to verify delivery and remove its own
objects. The remote endpoint uses the credentials in the Modal secret. Keep the
bucket and endpoint values consistent with the deployed output policy; changing
that policy requires redeployment.

In [ ]:
from __future__ import annotations

from r2_wait import R2WaitTool

from artisan.operations.examples import DataGenerator
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.schemas import (
    ComputeProvider,
    ModalComputeConfig,
    ToolEndpointDataPolicy,
)
from artisan.utils import env_or_dotenv, tutorial_setup
from artisan.visualization import inspect_step

env = tutorial_setup("modal_r2_outputs", clean=True)

In [ ]:
import os
import uuid

BUCKET = env_or_dotenv("ARTISAN_S3_BUCKET")
ENDPOINT = env_or_dotenv("ARTISAN_S3_ENDPOINT_URL")
assert BUCKET, "set ARTISAN_S3_BUCKET (see setup above)"
assert ENDPOINT, "set ARTISAN_S3_ENDPOINT_URL (see setup above)"

# ambient credentials for the verification cells at the end — the same
# variable shapes the worker's Modal Secret injects on the other side
for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"):
    os.environ.setdefault(key, env_or_dotenv(key) or "")
os.environ.setdefault("AWS_ENDPOINT_URL", ENDPOINT)

OUTPUT_STORE = f"s3://{BUCKET}/artisan-tutorial/{uuid.uuid4().hex[:8]}"
OUTPUT_POLICY = ToolEndpointDataPolicy(
    output_allowlist=(f"s3://{BUCKET}/artisan-tutorial", ENDPOINT),
)
print(f"outputs will be delivered under {OUTPUT_STORE}")

## Run on Modal and deliver to the bucket

Choose a unique `output_store` below the deployed prefix. The client checks the
request against its policy, and the endpoint checks it against the deployed
policy. `skip_cache=True` ensures rerunning the cell performs another remote
execution and delivery.

In [ ]:
pipeline = PipelineManager.create(
    name="modal_r2_outputs",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

gen = pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 2, "seed": 42},
)

wait = pipeline.run(
    operation=R2WaitTool,
    skip_cache=True,
    name="wait",
    inputs={"dataset": gen.output("datasets")},
    params={"seconds": 1},
    compute_provider=ComputeProvider(
        active="modal",
        modal=ModalComputeConfig(output_store=OUTPUT_STORE, data_policy=OUTPUT_POLICY),
    ),
)

result = pipeline.finalize()
assert result["overall_success"], result
assert wait.disposition is StepDisposition.EXECUTED
for completed_step in (gen, wait):
    artifacts = inspect_step(
        env.delta_root,
        completed_step.step_number,
        pipeline_run_id=pipeline.config.pipeline_run_id,
    )
    assert artifacts.height == 2, (completed_step.step_name, artifacts)
print("pipeline complete")

## What happened

Each input caused one endpoint call. The remote tool wrote a marker, uploaded
its output archive under our prefix, and returned a presigned download URL. The
client downloaded each archive and committed the resulting artifact. The checks
above verified fresh execution and two accepted outputs.

A presigned URL grants access to its object until it expires. See
[Object-store output delivery](../../how-to-guides/deploying-tool-endpoints.md#object-store-output-delivery)
for retention and external-client details.

## Verify the delivery

Two artifacts fanned out as two endpoint calls, so two tarballs landed
under the prefix. List them, and build a link to browse them in your
provider's web console:


In [ ]:
from urllib.parse import urlparse

import s3fs

fs = s3fs.S3FileSystem()
delivered = fs.find(OUTPUT_STORE.removeprefix("s3://"))
assert len(delivered) == 2, delivered
assert all(key.endswith(".tar.gz") for key in delivered)
for key in delivered:
    print(key)

host = urlparse(ENDPOINT).netloc
prefix = OUTPUT_STORE.removeprefix(f"s3://{BUCKET}/")
if host.endswith(".r2.cloudflarestorage.com"):
    account_id = host.split(".", 1)[0]
    console = f"https://dash.cloudflare.com/{account_id}/r2/default/buckets/{BUCKET}"
elif host.endswith(".amazonaws.com"):
    console = f"https://s3.console.aws.amazon.com/s3/buckets/{BUCKET}?prefix={prefix}/"
else:
    console = None  # self-hosted stores (MinIO, ...) serve their own console
print(
    "\nbrowse the delivered files:", console or f"open your store's console for {host}"
)

## Clean up

Artisan never deletes delivered tarballs — pair real destination
prefixes with a bucket lifecycle policy. For the tutorial prefix, remove
the objects directly:


In [ ]:
output_prefix = OUTPUT_STORE.removeprefix("s3://")
if fs.exists(output_prefix):
    fs.rm(output_prefix, recursive=True)
    print("removed", OUTPUT_STORE)

## Summary

- `output_store` on the modal config delivers a Modal-executed op's
  outputs beneath a prefix allowed by its baked `data_policy`, bypassing
  the 100 MB inline bound while retaining stored-archive safety budgets
- The worker uploads with its Modal Secret's credentials; consumers
  fetch through an allowed presigned-URL origin, credential-free
- Changing the allowed prefixes or origins requires redeploying; callers
  cannot widen policy through request fields
- Without `output_store`, behavior is unchanged: outputs return inline,
  bounded at 100 MB

## Next steps

- [Object-store output delivery](../../how-to-guides/deploying-tool-endpoints.md#object-store-output-delivery)
  — the full contract: endpoint policy, IAM scoping, lifecycle, external consumers
- [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md)
  — putting the pipeline's own storage (Delta tables, staging, files)
  on a bucket
- [Running on Modal](04-modal-execution.ipynb) — the endpoint model
  this builds on
- [Execution Flow](../../concepts/execution-flow.md) — how compute
  routing and output delivery fit the execution model